# Insurance Claim Prediction using Machine Learning

## Task 1 - Dataset Understanding

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: preprocessing, model, evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

In [ ]:
data = pd.read_csv("/content/insurance2.csv")

In [ ]:
# First 5 records
data.head()

In [ ]:
# Last 5 records
data.tail()

In [ ]:
# Number of rows and columns
print('Shape (rows, columns):', data.shape)
print('Rows   :', data.shape[0])
print('Columns:', data.shape[1])

In [ ]:
# Column names
print(list(data.columns))

In [ ]:
# Data types
data.dtypes

In [ ]:
# Missing values
data.isna().sum()

In [ ]:
# Target class balance
print(data['insuranceclaim'].value_counts())
print('\nClass proportions:')
print(data['insuranceclaim'].value_counts(normalize=True).round(3))

### Questions to Answer

1. **Which variables are numerical?** `age`, `bmi`, `children`, `charges` (continuous / count values).
2. **Which variables are categorical?** `sex`, `smoker`, `region` (stored as numeric codes), and the target `insuranceclaim` (binary).
3. **What is the target variable?** `insuranceclaim` (1 = claim, 0 = no claim).
4. **Is the target variable balanced?** Not perfectly - roughly 59% of policies have a claim (1) and 41% do not (0). The imbalance is mild, so accuracy is still meaningful, but precision/recall/F1 should also be checked.

## Task 2 - Exploratory Data Analysis

Univariate analysis of Age, BMI, Charges and Insurance claims.

In [ ]:
# Age
plt.figure(figsize=(7, 4))
sns.histplot(data['age'], bins=20, kde=True)
plt.title('Distribution of Age')
plt.show()

In [ ]:
# BMI
plt.figure(figsize=(7, 4))
sns.histplot(data['bmi'], bins=30, kde=True)
plt.title('Distribution of BMI')
plt.show()

In [ ]:
# Charges
plt.figure(figsize=(7, 4))
sns.histplot(data['charges'], bins=30, kde=True)
plt.title('Distribution of Charges')
plt.show()

In [ ]:
# Insurance claims
plt.figure(figsize=(5, 4))
sns.countplot(x='insuranceclaim', data=data)
plt.title('Insurance Claim Count (0 = No Claim, 1 = Claim)')
plt.show()

## Task 3 - Data Preprocessing

In [ ]:
X = data.drop(columns=['insuranceclaim'])  # Independent variables
y = data['insuranceclaim']                 # Target variable

print('Features (X) shape:', X.shape)
print('Target (y) shape  :', y.shape)

### Encode categorical variables

`sex` and `smoker` are already binary (0/1). `region` has 4 unordered categories, so it is one-hot encoded (dropping the first level) to avoid implying a false order between regions.

In [ ]:
X = pd.get_dummies(X, columns=['region'], prefix='region', drop_first=True, dtype=int)
X.head()

### Train-Test Split

80/20 split with `random_state` for reproducibility and `stratify=y` to keep the same claim ratio in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print('Training set:', X_train.shape, '| Test set:', X_test.shape)

### Feature Scaling

Only the continuous columns are scaled. The scaler is **fit on the training set only** and then used to transform both sets, so no information from the test set leaks into preprocessing.

In [ ]:
num_cols = ['age', 'bmi', 'children', 'charges']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

scaler = StandardScaler()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])  # fit on train only
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])        # transform only

X_train_scaled.head()

### Question

**Why should preprocessing transformations be learned from the training dataset rather than the complete dataset?**

The test set represents unseen, future data. If the scaler (mean / standard deviation) were learned from the complete dataset, information from the test set would influence the training pipeline - this is **data leakage**. The model would then be evaluated on data that has indirectly been seen, giving over-optimistic performance that will not hold in production. Learning the transformations from the training data only and merely applying them to the test data keeps the evaluation honest.

## Task 4 - Build Machine Learning Models

### Selected algorithm: Logistic Regression

The target is binary (claim / no claim), and Logistic Regression outputs class probabilities, is interpretable and is a strong baseline for small tabular data.

In [ ]:
# 1. Train the model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)
print('Model trained:', model)

In [ ]:
# 2. Generate predictions
train_predictions = model.predict(X_train_scaled)
test_predictions = model.predict(X_test_scaled)

# 3. Generate prediction probabilities
test_probabilities = model.predict_proba(X_test_scaled)[:, 1]  # probability of a claim

pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': test_predictions[:10],
    'Probability of claim': test_probabilities[:10].round(3),
})

### 4. Evaluate the model

In [ ]:
print(f'Training accuracy: {accuracy_score(y_train, train_predictions):.4f}')
print(f'Test accuracy    : {accuracy_score(y_test, test_predictions):.4f}')
print(f'Precision        : {precision_score(y_test, test_predictions):.4f}')
print(f'Recall           : {recall_score(y_test, test_predictions):.4f}')
print(f'F1-score         : {f1_score(y_test, test_predictions):.4f}')
print(f'ROC-AUC          : {roc_auc_score(y_test, test_probabilities):.4f}')
print('\nFull classification report:\n')
print(classification_report(y_test, test_predictions, target_names=['No Claim', 'Claim']))

In [ ]:
cm = confusion_matrix(y_test, test_predictions)
print('Confusion matrix:\n', cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Claim', 'Claim'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix (Test Set)')
plt.show()